## **설정**

In [ ]:
## Google Drive Amount
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
## Import libaries
import pandas as pd
import numpy as np

import os

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
## Setting
base_path = '/content/drive/MyDrive/CS2/'
output_path = os.path.join(base_path, 'SimulData')

## **데이터 불러오기**

In [ ]:
## Load the data
# 1. Prediction Result : ticker, date, permno, pred_ret, true_ret
file_path = os.path.join(base_path, 'PredData', '03_CatBoost_M2010_F2020')

result_2018 = pd.read_csv(os.path.join(file_path, 'result_2018.csv'))
result_2019 = pd.read_csv(os.path.join(file_path, 'result_2019.csv'))
result_2020 = pd.read_csv(os.path.join(file_path, 'result_2020.csv'))
result_2021 = pd.read_csv(os.path.join(file_path, 'result_2021.csv'))

In [ ]:
# 2. Firm Top 1000 데이터
firm_top1000 = pd.read_csv(os.path.join(base_path, 'myData', 'firm_top1000.csv'))

# 3. macroeco 데이터
macroeco = pd.read_csv(os.path.join(base_path, 'myData', 'macroeco_data.csv'))
macroeco_lag1 = pd.read_csv(os.path.join(base_path, 'myData', 'macroeco_data_lag1.csv'))

## **데이터 병합**

In [ ]:
df_all = pd.concat([result_2018, result_2019, result_2020, result_2021], axis=0)
df_all.reset_index(drop=True, inplace=True)

df_all['date'] = pd.to_datetime(df_all['date'])
df_all['Year'] = df_all['date'].dt.year
df_all['Month'] = df_all['date'].dt.month

print(f"전체 데이터 크기: {df_all.shape}")

전체 데이터 크기: (46243, 7)


In [ ]:
## save
df_all.to_csv(os.path.join(output_path, 'result_all.csv'), index=False)

### **시가 총액**

In [ ]:
# 병합을 위한 임시 데이터
temp_me = firm_top1000[['permno', 'date', 'me']].copy()
temp_me['date'] = pd.to_datetime(temp_me['date'])
temp_me['Year'] = temp_me['date'].dt.year
temp_me['Month'] = temp_me['date'].dt.month
temp_me.drop(['date'], axis=1, inplace=True)

In [ ]:
# 병합
df_all_with_me = pd.merge(df_all, temp_me, on=['permno','Year', 'Month'], how='left')
df_all_with_me.drop(['Year', 'Month'], axis=1, inplace=True)

In [ ]:
df_all_with_me

,ticker,date,permno,pred_ret,true_ret,me
0,AAL,2018-01-31,21020,-0.001045,0.044013,2.474068e+07
1,PNW,2018-01-31,27991,0.023113,-0.053240,9.517161e+06
2,AAN,2018-01-31,10517,0.014096,0.026098,2.819905e+06
3,ABT,2018-01-31,20482,0.014265,0.094095,9.933610e+07
4,AMD,2018-01-31,61241,0.013793,0.336576,9.940760e+06
...,...,...,...,...,...,...
46238,CBRE,2021-12-31,90199,-0.016897,0.135398,3.198403e+07
46239,ALNY,2021-12-31,90178,-0.021726,-0.077367,2.198266e+07
46240,ESI,2021-12-31,14406,-0.018475,0.061653,5.660394e+06
46241,APG,2021-12-31,19317,-0.002886,0.105534,5.236009e+06


In [ ]:
df_all_with_me.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46243 entries, 0 to 46242
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   ticker    46243 non-null  object        
 1   date      46243 non-null  datetime64[ns]
 2   permno    46243 non-null  int64         
 3   pred_ret  46243 non-null  float64       
 4   true_ret  46243 non-null  float64       
 5   me        46243 non-null  float64       
dtypes: datetime64[ns](1), float64(3), int64(1), object(1)
memory usage: 2.1+ MB


In [ ]:
## save
df_all_with_me.to_csv(os.path.join(output_path, 'result_all_with_me.csv'), index=False)

## **beta 계수**

### **필요한 데이터 병합**

In [ ]:
temp_f = firm_top1000[['date', 'mom1m']].copy()

temp_f['mom1m'] = 100*temp_f['mom1m']

temp_f['date'] = pd.to_datetime(temp_f['date'])
temp_f['Year'] = temp_f['date'].dt.year
temp_f['Month'] = temp_f['date'].dt.month

In [ ]:
temp_m = macroeco[['sasdate', 'S&P 500']].copy()

temp_m['dSP500'] = temp_m['S&P 500'].pct_change()

temp_m['date'] = pd.to_datetime(temp_m['sasdate'])
temp_m.drop(['sasdate'], axis=1, inplace=True)
temp_m['Year'] = temp_m['date'].dt.year
temp_m['Month'] = temp_m['date'].dt.month
temp_m.drop(['date'], axis=1, inplace=True)

In [ ]:
temp_fm = pd.merge(temp_f, temp_m, on=['Year', 'Month'], how='left')
temp_fm = temp_fm[['date', 'Year', 'Month', 'mom1m','S&P 500', 'dSP500']]

In [ ]:
temp_fm.head()

,date,Year,Month,mom1m,S&P 500,dSP500
0,1996-08-31,1996,8,-7.692308,662.68,0.028894
1,1996-09-30,1996,9,16.666667,674.88,0.018410
2,1996-10-31,1996,10,-16.666667,701.46,0.039385
3,1996-11-30,1996,11,-11.428571,735.67,0.048770
4,1996-12-31,1996,12,-6.451613,743.25,0.010304


### **Rolling OLS**

In [ ]:
import statsmodels.api as sm
from statsmodels.regression.rolling import RollingOLS

In [ ]:
y = temp_fm['mom1m']
X = sm.add_constant(temp_fm['dSP500'])

# 3. Rolling OLS 설정 (window=60 : 과거 60개월 데이터를 쓰겠다는 뜻)
# min_nobs=30 : 최소 30개 데이터는 있어야 계산 시작
rolling_model = RollingOLS(y, X, window=60, min_nobs=30)
rolling_res = rolling_model.fit()

In [ ]:
# 4. 시점별 베타 추출
# params['dSP500']에 매달 그 시점 기준의 베타가 저장됨
beta_series = rolling_res.params['dSP500']

In [ ]:
# 5. [매우 중요] Shift(1) 적용 (Leakage 방지 핵심)
# rolling_res로 나온 '2017-12-31'의 베타값은 2017년 12월까지의 데이터로 구한 것임.
# 이 값은 '2018-01'을 예측할 때 써야 함. 따라서 한 칸 밑으로 내려야(shift) 함.
temp_fm['Beta'] = beta_series.shift(1)

temp_fm['sigma'] = temp_fm['dSP500'].rolling(window=60).std()

In [ ]:
temp_fm_filtered = temp_fm[(temp_fm['Year'] >= 2013) & (temp_fm['Year'] <= 2021)].copy()
temp_fm_filtered.reset_index(drop=True, inplace=True)
temp_fm_filtered.drop(['date'], axis=1, inplace=True)

In [ ]:
temp_fm_filtered

,Year,Month,mom1m,S&P 500,dSP500,Beta,sigma
0,2013,1,7.182026,1480.40,0.040857,45.939522,0.025917
1,2013,2,9.379581,1512.31,0.021555,52.686318,0.025683
2,2013,3,-7.369360,1550.83,0.025471,55.965214,0.025666
3,2013,4,1.470592,1570.70,0.012812,48.398502,0.025608
4,2013,5,-1.262813,1639.84,0.044019,48.513281,0.025285
...,...,...,...,...,...,...,...
104313,2021,8,8.902123,4454.21,0.020739,5.952955,0.017995
104314,2021,9,0.917613,4445.54,-0.001946,6.198374,0.017995
104315,2021,10,-5.093859,4460.71,0.003412,29.528820,0.017995
104316,2021,11,11.491705,4667.39,0.046333,28.627361,0.017995


In [ ]:
temp_all_with_me = pd.merge(df_all, temp_me, on=['permno','Year', 'Month'], how='left')

In [ ]:
df_all_with_me_beta = pd.merge(temp_all_with_me, temp_fm_filtered, on=['Year', 'Month'], how='left')

In [ ]:
df_all_with_me_beta

,ticker,date,permno,pred_ret,true_ret,Year,Month,me,mom1m,S&P 500,dSP500,Beta,sigma
0,AAL,2018-01-31,21020,-0.001045,0.044013,2018,1,2.474068e+07,-3.063253,2789.80,0.047089,70.835425,0.009010
1,AAL,2018-01-31,21020,-0.001045,0.044013,2018,1,2.474068e+07,29.775959,2789.80,0.047089,-13.751002,0.017214
2,AAL,2018-01-31,21020,-0.001045,0.044013,2018,1,2.474068e+07,3.050103,2789.80,0.047089,101.611807,0.022089
3,AAL,2018-01-31,21020,-0.001045,0.044013,2018,1,2.474068e+07,5.726398,2789.80,0.047089,54.593913,0.025614
4,AAL,2018-01-31,21020,-0.001045,0.044013,2018,1,2.474068e+07,-1.237621,2789.80,0.047089,40.263518,0.028297
...,...,...,...,...,...,...,...,...,...,...,...,...,...
44556714,SNOW,2021-12-31,19654,0.003965,-0.004116,2021,12,1.023511e+08,0.746150,4674.77,0.001581,31.899713,0.017977
44556715,SNOW,2021-12-31,19654,0.003965,-0.004116,2021,12,1.023511e+08,-23.025666,4674.77,0.001581,-28.841808,0.017977
44556716,SNOW,2021-12-31,19654,0.003965,-0.004116,2021,12,1.023511e+08,-18.292682,4674.77,0.001581,-9.933473,0.017977
44556717,SNOW,2021-12-31,19654,0.003965,-0.004116,2021,12,1.023511e+08,8.815786,4674.77,0.001581,19.735112,0.017977


In [ ]:
df_all_with_me_beta.drop(['Year', 'Month', 'mom1m', 'dSP500'], axis=1, inplace=True)

In [ ]:
df_all_with_me_beta

,ticker,date,permno,pred_ret,true_ret,me,S&P 500,Beta,sigma
0,AAL,2018-01-31,21020,-0.001045,0.044013,2.474068e+07,2789.80,70.835425,0.009010
1,AAL,2018-01-31,21020,-0.001045,0.044013,2.474068e+07,2789.80,-13.751002,0.017214
2,AAL,2018-01-31,21020,-0.001045,0.044013,2.474068e+07,2789.80,101.611807,0.022089
3,AAL,2018-01-31,21020,-0.001045,0.044013,2.474068e+07,2789.80,54.593913,0.025614
4,AAL,2018-01-31,21020,-0.001045,0.044013,2.474068e+07,2789.80,40.263518,0.028297
...,...,...,...,...,...,...,...,...,...
44556714,SNOW,2021-12-31,19654,0.003965,-0.004116,1.023511e+08,4674.77,31.899713,0.017977
44556715,SNOW,2021-12-31,19654,0.003965,-0.004116,1.023511e+08,4674.77,-28.841808,0.017977
44556716,SNOW,2021-12-31,19654,0.003965,-0.004116,1.023511e+08,4674.77,-9.933473,0.017977
44556717,SNOW,2021-12-31,19654,0.003965,-0.004116,1.023511e+08,4674.77,19.735112,0.017977


In [ ]:
df_all_with_me_beta.isna().sum()

,0
ticker,0
date,0
permno,0
pred_ret,0
true_ret,0
me,0
S&P 500,0
Beta,0
sigma,0


In [ ]:
## save
df_all_with_me_beta.to_csv(os.path.join(output_path, 'result_all_with_me_beta.csv'), index=False)